# Estimation Universe (ESTU) — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the **Estimation Universe (ESTU)** used by every USE4 style factor — Beta, NLB, NLS, Size, Residual Volatility, Liquidity, Momentum, Earnings Yield, Book-to-Price, Growth, Leverage, Dividend Yield. The PDFs say everything is built off MSCI USA IMI; this notebook turns that into something we can actually run from Sharadar data.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDFs do not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard MSCI IMI practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- `SHARADAR_TICKERS.csv` (Sharadar ticker master) — security master with `permaticker`, `category`, `exchange`, `isdelisted`, `firstpricedate`, `lastpricedate`, …
- `SHARADAR_SEP.csv` (Sharadar daily prices) — daily OHLCV, `marketcap`, `volume`, `closeunadj`

**Deliverable:** A parquet file `data/out/estu_monthly.parquet` keyed by `(permaticker, signal_date)` containing the ESTU membership flag and supporting fields. This file is consumed by every USE4 descriptor we build later.

**Why ESTU matters more than any single descriptor.** The ESTU choice determines:
- which stocks contribute to standardization moments (cap-weighted mean, equal-weighted std)
- which stocks define the cap-weighted market benchmark used by Beta
- which stocks enter the cross-sectional WLS regression for factor-return estimation
- which stocks orthogonalization regressions operate on

Get ESTU wrong and **every** style factor is biased in a correlated way. Get it right and the factor library is internally consistent.

---

### Critical reading

The USE4 PDFs specify ESTU in **two places**:
- **Empirical Notes, Section 3.1 (p.7)** — what ESTU is (MSCI USA IMI, 99% float target)
- **Methodology Notes, Section 3.1 (p.13)** — how ESTU is used (cross-sectional regression with sqrt-mcap weights)

The PDFs **do not** define MSCI USA IMI construction itself. For that you have to reference the [*MSCI Global Investable Market Indexes Methodology* (GIMI) document](https://www.msci.com/index-methodology), which is publicly available but proprietary in its exact thresholds. The default pipeline below approximates MSCI IMI's three principles (representation, liquidity, stability) using fields available in Sharadar.

## §1. The USE4 ESTU spec — verbatim quotes

### 1a. Empirical Notes, Section 3.1 (p.7) — what ESTU is

> *"The estimation universe is the set of stocks used for estimating the factor model. Although the model can be used to forecast risk for stocks outside this set, **selection of an appropriate estimation universe is a critical decision in building a sound risk model**. A well-constructed estimation universe should be:*
>
> *(a) **sufficiently broad** to accurately represent the investment opportunity set of the manager, without being so wide as to include illiquid stocks that may introduce spurious return relationships into the model;*
>
> *(b) **sufficiently liquid** so that estimated factor returns capture meaningful market activity and not idiosyncratic noise from thinly traded stocks; and*
>
> *(c) **reasonably stable in composition**, so that factor exposures and returns are not unduly affected by stocks frequently entering and leaving the universe.*
>
> *The USE4 estimation universe utilizes the **MSCI USA Investable Markets Index (USA IMI)**, which aims to reflect the full breadth of investment opportunities within the US market by targeting **99 percent of the float-adjusted market capitalization**. The MSCI methodology incorporates a thorough framework for size segmentation, as well as buffer rules that promote stability of index constituents. We refer the reader to the MSCI Global Investable Market Indexes Methodology document for further details."*

### 1b. Methodology Notes, Section 3.1 (p.13) — how ESTU is used in cross-sectional regression

> *"Factor returns in USE4 are estimated using weighted least-squares regression, assuming that the variance of specific returns is inversely proportional to the square root of total market capitalization. This regression-weighting scheme reflects the empirical observation that the idiosyncratic risk of a stock decreases as the market capitalization of the firm increases. **Only stocks in the estimation universe are used to estimate the factor returns**; stocks outside the estimation universe receive exposures and forecasts but do not contribute to factor-return estimation."*

### 1c. Methodology Notes, Section 2.3 (p.9) — ESTU defines the standardization moments

> *"… where μ_l is the **cap-weighted mean of the descriptor (within the estimation universe)**, and σ_l is the **equal-weighted standard deviation**. We adopt the convention of standardizing using the cap-weighted mean so that a well-diversified cap-weighted portfolio has approximately zero exposure to all style factors."*

### 1d. Empirical Notes, Section 3.1 (p.7) — distinction between estimation universe and coverage universe

> *"Although the model can be used to forecast risk for stocks outside this set …"*

Implicit but important: there are **two universes** in USE4:
- **Estimation universe (ESTU)** — used for factor-return estimation, standardization moments, market benchmark construction
- **Coverage universe** — every stock that gets an exposure value and a risk forecast (this is the *output* universe, much larger than ESTU)

Sharadar's coverage universe is roughly every US-listed equity-like security: ~18,000 unique permatickers historically, ~9,000 active in recent years. Our ESTU should be ~2,500–3,000 names per date.

---

**That is all the USE4 PDFs say about ESTU.** They defer the actual rules to the MSCI GIMI document. The three principles — *representation, liquidity, stability* — are the binding constraints; the rest is implementation choices to be documented in §13.

## §2. MSCI USA IMI background — the three principles

Since the USE4 PDFs defer to MSCI GIMI, this section captures what the IMI methodology actually does, paraphrased from the public MSCI GIMI document. We're not trying to clone MSCI IMI exactly — we're matching its three principles using Sharadar fields.

### 2a. Principle 1 — Representation (target 99% float-adjusted mcap)

MSCI IMI covers approximately **99% of the free-float-adjusted market capitalization** of the US equity universe, split into three size segments:
- **Large cap** — top 70% of float-adjusted mcap (~ top 300 names)
- **Mid cap** — next 15% (~ next 400 names, so ~700 cumulative)
- **Small cap** — next 14% (~ next 1,700 names, so ~2,400 cumulative). The remaining ~1% is micro/nano and is excluded.

**Net result:** ~2,400–2,500 names in MSCI USA IMI as of recent years.

### 2b. Principle 2 — Liquidity (Annual Traded Value Ratio)

MSCI uses an **Annual Traded Value Ratio (ATVR)** that's roughly:
$$\text{ATVR} = \frac{\text{annualized 3-month median daily dollar volume}}{\text{security float mcap}}$$

Thresholds:
- New additions: ATVR ≥ 20%
- Existing constituents stay in until ATVR < 10% (asymmetric buffer)
- Combined with **frequency of trading** (% of days with at least one trade)

**Our proxy:** trailing 63-day median dollar volume in absolute terms (e.g., > $1M/day). Less precise but uses only Sharadar's `volume` and `closeunadj`.

### 2c. Principle 3 — Stability (buffer rules)

MSCI uses **asymmetric size buffers**: a stock has to grow into a larger segment by a wider margin than it needs to shrink out. Specifically:
- Mid → Large requires mcap > 1.8× large-cap cutoff
- Large → Mid only requires mcap < 1.0× large-cap cutoff
- Similar 1.8×/1.0× rules at the small-cap boundary

**Net result:** monthly index turnover is typically < 1–2%, vs ~5–10% with naive top-N ranking.

**Our proxy:** addition threshold = top 3000 by mcap, deletion threshold = drop only if rank > 3500. Asymmetric ranks mimic the asymmetric buffer logic.

### 2d. What we do NOT have from Sharadar

- **Float-adjusted mcap** — Sharadar `marketcap` is total mcap (shares outstanding × close). True float would require a free-float ratio (often 0.7–0.95 for liquid US names but lower for founder-controlled and ADR-style listings). We'll use total mcap and document the gap.
- **ATVR (proper definition)** — would require float, which we don't have. Substitute absolute dollar volume threshold.
- **REIT / business-trust / closed-end-fund distinctions** — Sharadar `category` does separate REITs but inclusion in IMI is partial. We'll include them with a flag for downstream selective exclusion.

These gaps are documented as ❓ NOT-IN-PDF decisions in §13.

## §3. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths, parameters                            │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load Sharadar ticker master + daily prices                   │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build monthly signal_date calendar (end-of-month)            │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 3:  Hard filters (security type, exchange, delisted, mcap > 0)   │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  Liquidity screen (63-day median dollar volume)               │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Size segmentation (top-N or cumulative 99% mcap cutoff)      │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Stability buffer (asymmetric add/drop thresholds)            │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Assemble final panel + write parquet                         │
├─────────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                     │
└─────────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any signal_date `t`, the only data permitted is data dated `≤ t`. The 63-day liquidity window only contains observations dated `≤ t`. Use `bisect.bisect_right` on a sorted trading-day list.

**Membership churn:** ESTU is rebuilt **per signal_date** (end of each month). With the buffer rule of Stage 6, month-over-month membership should be 95%+ stable.

## §4. Output schema — what the worker delivers

**File:** `data/out/estu_monthly.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `ticker` | String | Exchange ticker symbol as of signal_date (PIT-correct; reflects renames, e.g. FB → META) |
| `in_estu` | Bool | **The deliverable.** True if the stock is in ESTU on this date. |
| `mcap` | Float64 | Market cap on signal_date (in dollars — see §6 unit note) |
| `dollar_volume_63d` | Float64 | Trailing 63-day median dollar volume (used by liquidity screen); null if < 42 observations in the 63-day window |
| `mcap_rank` | UInt32 | Rank by mcap within date-cross-section among stocks passing the hard filter **and** the 10% ATVR retain threshold (1 = largest); null for stocks failing either gate |
| `passed_hard_filter` | Bool | Passed §7 hard filters |
| `passed_liquidity` | Bool | Passed §8 liquidity screen (the 20% **add** ATVR threshold; stocks failing it but passing the 10% retain threshold can still be `in_estu=True` via the stability buffer) |
| `size_segment` | String | `large` / `mid` / `small` / `micro` (only when `INCLUDE_MICRO=True`) / `micro_excluded` / `ineligible` |
| `retained_via_buffer` | Bool | True if kept in ESTU this month solely by the stability buffer — existing member whose rank slipped into 3,001–3,500 |
| `entered_estu` | Bool | True if this stock was not in ESTU last month and entered this month; always False on the first signal date |

*Note:* the design sketches in the stage cells below predate implementation and refer
to a single `entered_via_buffer` flag; the implemented build
splits the buffer diagnostics into `retained_via_buffer` and `entered_estu` as
documented above. This table reflects the delivered file.

**Sizing expectation:**
- ~2500–3000 rows with `in_estu=True` per signal_date in recent years
- ~9000–12000 rows total per signal_date across coverage universe (every stock that exists on that date, with diagnostic fields filled in even if it failed an upstream filter)

**Why keep the diagnostic columns?** They let downstream consumers swap in alternative ESTUs without rerunning this pipeline — e.g., "give me top 1500 by mcap that passed liquidity" can be answered by filtering this file directly.

## §5. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

```python
# ───── Parameters  ──────────────────────────────────────────────────────
ALLOWED_CATEGORIES    = {"Domestic Common Stock",
                         "Domestic Common Stock Primary Class"}
ALLOWED_EXCHANGES     = {"NYSE", "NASDAQ", "NYSEMKT"}
INCLUDE_MICRO         = False
MCAP_MIN_USD          = 50_000_000.0
MICRO_MCAP_THRESHOLD  = 300_000_000.0
LIQUIDITY_WINDOW_DAYS = 63        # trailing window for median dollar volume
LIQUIDITY_MIN_SAMPLES = 42        # min obs in window (excludes fresh IPOs)
ATVR_ADD_THRESHOLD    = 0.20      # ATVR proxy to ENTER the ESTU
ATVR_RETAIN_THRESHOLD = 0.10      # ATVR proxy to REMAIN in the ESTU
BUFFER_ADD_RANK_LE    = 3000      # hysteresis buffer: add at rank ≤ 3000
BUFFER_DROP_RANK_GT   = 3500      # drop only past rank 3500
CALENDAR_START        = date(1998, 1, 1)   # 1yr warm-up before 1999 signals
```


## §6. STAGE 1 — Load Sharadar ticker master + daily prices

**Inputs:**
- `SHARADAR_TICKERS.csv` (Sharadar ticker master) — the security master. Needed columns: `permaticker`, `category`, `exchange`, `isdelisted`, `firstpricedate`, `lastpricedate`.
- `SHARADAR_SEP.csv` (Sharadar daily prices) — daily prices. Needed columns: `permaticker`, `date`, `closeunadj`, `closeadj`, `volume`, `marketcap`.

**Outputs:**
- `daily` — polars DataFrame keyed by `(permaticker, date)` with computed `dollar_volume = closeunadj * volume`
- `tickers` — polars DataFrame keyed by `permaticker` with `category`, `exchange`, `isdelisted`

### Unit gotcha — Sharadar `marketcap`

**Sharadar `marketcap` is denominated in millions of USD, not dollars.**

💡 **DEFAULT for this notebook:** convert `marketcap` to dollars at load time: `marketcap_usd = marketcap * 1_000_000`. Downstream code should never need to think about the unit again.

### `closeunadj` vs `closeadj` for dollar volume

Dollar volume should use **unadjusted** close × **unadjusted** volume (i.e., raw shares traded × raw price). Sharadar `volume` is unadjusted by default; pair it with `closeunadj`.

### `category` mapping

Sharadar categories that are equity-like and US-domestic:
```
Domestic Common Stock
Domestic Common Stock Primary Class
Domestic Common Stock Secondary Class
Domestic Common Stock REIT          ← include if INCLUDE_REITS
```

Categories to **exclude**:
```
Domestic Preferred
ADR Common Stock                      (foreign issuer; not in USA IMI)
ADR Common Stock Primary Class
ADR Common Stock REIT                 (foreign REIT)
Closed-Ended Fund                     (not equity)
ETF                                   (not equity)
ETN, ETD, etc.
Domestic Common Stock Warrant
```

🧪 **VALIDATE:**
- `tickers.shape[0]` ~ 18,000–22,000 (Sharadar full security master)
- `daily.shape[0]` ~ 50M+ rows for full historical sample
- Spot check: AAPL/MSFT have `category = "Domestic Common Stock Primary Class"`, `exchange = "NASDAQ"`, `isdelisted = False`

## §7. STAGE 2 — Monthly signal_date calendar

ESTU is rebuilt monthly. Each signal_date is the **last trading day of the month**.

### Why end-of-month

- **Aligns with industry convention.** Most factor research and risk models rebuild at month-end.
- **MSCI rebalances quarterly with monthly index reviews.** End-of-month is the natural alignment.
- **The downstream factor notebooks (Beta, NLB, NLS) all assume monthly signal_dates.** Stay consistent.

### Why not quarterly

MSCI's actual IMI rebalances are quarterly (with semi-annual index reviews). We use monthly because:
1. Style factor exposures change month-to-month even if membership doesn't
2. Beta, momentum, and short-term style descriptors need monthly refresh
3. The stability buffer in Stage 6 makes the *actual* monthly turnover very low (<2%)

🧪 **VALIDATE:**
- One signal_date per month from `CALENDAR_START` to last available trading day
- Each signal_date is in `trading_days` (not a weekend or holiday)
- Spot check: Jan 2020 signal_date = 2020-01-31 (Friday), Feb 2020 = 2020-02-28 (Friday)

## §8. STAGE 3 — Hard filters (security type, exchange, delisted, mcap > 0)

For each `signal_date t`, build the universe of *eligible* securities — those that could possibly be in ESTU before any size/liquidity ranking.

❓ **NOT IN PDF:** the USE4 PDFs defer to MSCI IMI; MSCI IMI defines "US securities" by domicile, primary listing, and security type. The defaults below approximate this with Sharadar fields.

### Filter logic per signal_date

**1. Stock exists in `daily` on signal_date with positive mcap.** This handles delistings and not-yet-listed names. Use:
```
snap_t = daily.filter(pl.col("date") == t).filter(pl.col("mcap_usd") > 0)
```

**2. Security category is in `ALLOWED_CATEGORIES` (and possibly REITs).** Join `snap_t` against `tickers` and filter `category`.

**3. Exchange is in `ALLOWED_EXCHANGES`.** NYSE, NASDAQ, NYSEMKT (American Stock Exchange). Excludes OTC and BATS.

**4. Not delisted.** `isdelisted = False`. **Important PIT subtlety:** `isdelisted` is the *current* state from the ticker master. A stock that delisted in 2015 but was active in 2010 has `isdelisted=True` today. We need a **PIT-correct delisted check**: a security is delisted as of signal_date `t` if `lastpricedate < t`. Use:
```
delisted_as_of_t = pl.col("lastpricedate") < t
```
However, this is implicitly handled by the "stock exists in daily on signal_date" condition (Sharadar stops emitting prices after delist). So filter #1 alone is sufficient, and the `isdelisted` field becomes informational rather than a filter.

💡 **DEFAULT:** rely on filter #1 (stock has a row in `daily` with positive mcap on `t`) and don't use `isdelisted` as a filter. Use `isdelisted` only for diagnostics.

**5. Price floor (optional):** some risk-model builds drop sub-$5 stocks to avoid penny-stock noise. MSCI IMI does not have a hard price floor; it has a liquidity floor (ATVR), which catches penny stocks indirectly.

💡 **DEFAULT:** no price floor — let the liquidity screen in Stage 4 handle it.

🧪 **VALIDATE:**
- After hard filters, ~3,500–5,000 names eligible per recent date (before liquidity & size screens)
- Spot check: AAPL, MSFT, GOOG, JPM all pass on every date they were public
- Spot check: Lehman Brothers `passed_hard_filter = True` from IPO date through 2008-09-15, then drops out

## §9. STAGE 4 — Liquidity screen (63-day median dollar volume)

❓ **NOT IN PDF.** USE4 says only "sufficiently liquid". MSCI IMI defines ATVR (Annual Traded Value Ratio) which depends on float, which we don't have.

💡 **DEFAULT — absolute dollar-volume threshold:**

For each `(permaticker, signal_date t)`, compute the trailing-63-day **median** dollar volume from `daily`:
$$\text{DV}_{i,t}^{63} = \text{median}_{\tau \in [t-63, t]}\big(\text{closeunadj}_{i,\tau} \cdot \text{volume}_{i,\tau}\big)$$

Threshold: $\text{DV}_{i,t}^{63} \geq \$1{,}000{,}000$ (= `LIQUIDITY_MIN_DOLLAR_VOL`).

### Why median, not mean

Earnings days and one-off catalysts blow up the mean. Median is robust to a handful of spikes. MSCI uses ADTV (mean) for ATVR but applies it on a 3-month rolling basis — for our absolute threshold, median is the safer choice.

### Why 63 days

- ~3 months of trading days, matches MSCI's 3-month ADTV window
- Also matches Beta's exponential half-life (consistency with downstream factors)
- Long enough to smooth noise, short enough to drop newly-illiquid names quickly

### Implementation note

Naive per-date filtering of `daily` is O(N · M) where N = trading days and M = securities. With ~25 years of data this is feasible but slow. Two faster options:

**(a) Daily rolling median, then filter to signal_dates.** Compute `dollar_volume_63d_median` for every (permaticker, date) using a rolling window expression. Then at signal_date selection, pick the row matching `date == signal_date`. Polars handles rolling medians natively via `rolling_median_by`.

**(b) Per-signal-date computation.** Inside the monthly loop, for each `t`, build the 63-day window of `daily` and group-by-permaticker to get the median. Simpler to reason about but slower if done naively.

💡 **DEFAULT:** option (a) — compute once for all dates, look up at signal_dates.

### Newly-listed stocks

❓ **What happens to stocks with < 63 days of trading history?**

**MSCI IMI requires** at least 3 months (~63 days) of trading history before adding a stock. This is a *deliberate* delay to ensure stable liquidity measurement.

💡 **DEFAULT:** require at least 42 (~ 2/3 of 63) trading days of history with non-null dollar volume before computing the median. Stocks with fewer days get `passed_liquidity = False`, automatically excluding fresh IPOs for ~2 months.

🧪 **VALIDATE:**
- After liquidity screen, ~3,000–4,000 names eligible per recent date
- Spot check: a known illiquid micro-cap should fail; a known liquid small-cap should pass
- IPOs from the last month should fail (insufficient observations)
- IPOs from 3+ months ago should pass (assuming the offering was well-traded)

## §10. STAGE 5 — Size segmentation (cap target)

Among the universe that passed hard filters AND liquidity, pick the top names by market cap to hit a coverage target.

✅ **PDF SPEC:** USE4 says ESTU "targets 99 percent of the float-adjusted market capitalization". That's the cumulative-fraction approach.

❓ **NOT IN PDF (and not in MSCI GIMI numerically):** what exactly the per-segment cutoffs are. Two valid implementations:

### Option A — Cumulative mcap fraction (literal USE4 spec)

Per signal_date `t`:
1. Sort eligible stocks (passed hard + liquidity) by `mcap_usd` descending
2. Compute cumulative mcap as fraction of total
3. Include all stocks where cumulative fraction ≤ `CUM_MCAP_FRAC` (default 0.99)
4. Note: this gives a *variable* number of names per date — typically 2,000–2,800 depending on market concentration

### Option B — Top-N (simpler, easier to reason about)

Per signal_date `t`:
1. Sort eligible stocks by `mcap_usd` descending
2. Include top `TOP_N_INCLUDE` (default 3000)
3. Gives a *fixed* number of names per date

💡 **DEFAULT — Option B (top-N).** Reasons:
- Stable count = simpler debugging
- 3000 is empirically close to what Option A gives in recent years
- Easier to reason about cross-sectional regression sample size

### Size segments (informational, not used for inclusion)

Even if we use Option B, label the size segment per MSCI IMI convention so downstream consumers can subset:
- **Large cap:** top 300
- **Mid cap:** ranks 301–700
- **Small cap:** ranks 701–3000
- **Micro/excluded:** rank > 3000

These segment labels are written into `size_segment` for downstream use.

🧪 **VALIDATE:**
- Option B: exactly `min(eligible_count, TOP_N_INCLUDE)` names per date with `passed_size_target = True`
- Option A: typically 2,000–2,800 per date in recent years; can dip below 2,000 in crisis periods (e.g., March 2009)
- The lowest-mcap name passing the size target should have mcap ~$300M–$1B in recent years (the small-cap floor)

## §11. STAGE 6 — Stability buffer (asymmetric add/drop)

❓ **NOT IN PDF numerically.** USE4 says "reasonably stable in composition". MSCI IMI uses 1.8×/1.0× size buffers on mcap thresholds.

💡 **DEFAULT — asymmetric rank buffer:**
- **Addition rule:** a stock is added to ESTU on month `t` only if its `mcap_rank` ≤ `BUFFER_ADD_RANK_LE` (default 3000) **AND** it was either:
  - already in ESTU on month `t-1`, or
  - newly meeting all eligibility criteria (passed_hard_filter, passed_liquidity, mcap_rank ≤ 3000)
- **Deletion rule:** a stock currently in ESTU is dropped on month `t` only if its `mcap_rank` > `BUFFER_DROP_RANK_GT` (default 3500) **OR** it fails the hard-filter or liquidity gates.

**The asymmetry:** stocks must rank in the top 3000 to enter, but a current member is kept until it falls below rank 3500. The ~500-rank hysteresis band prevents borderline stocks from flipping in/out every month.

### Why a buffer at all

Without a buffer, ~5–10% of ESTU turns over each month, just from stocks oscillating around the rank-3000 boundary. With the buffer, turnover drops to ~1–2%/month. This matters for:
- **Factor stability.** Cross-sectional standardization uses ESTU members. Bouncing members produce bouncing standardization moments, which produce bouncing factor exposures, which inflates the apparent volatility of style factors.
- **Industry/sector mix.** Sustained over a year, a 5%/month turnover rate would shift the sector composition meaningfully each year. A 1–2% rate is much closer to the real-economy pace of company evolution.
- **Beta estimation.** Beta uses 252 days of history. If members flip in and out, the cap-weighted-ESTU market benchmark itself is jumpy.

### State requirement

Implementing the buffer requires keeping **last month's ESTU membership** in scope when computing this month's. Either:
- Loop over `monthly_cal` in chronological order, carrying `prev_estu` forward (clean, slower)
- Two-pass: compute candidate ESTU (strict top-N) per date, then apply buffer in a second pass that has access to neighboring dates (vectorized, harder to debug)

💡 **DEFAULT:** chronological loop. ~330 dates is small; loop overhead doesn't matter.

### Bootstrap problem

On the very first signal_date, there is no `prev_estu`. Initialize with strict top-N for that date only. The buffer kicks in starting on month 2.

🧪 **VALIDATE:**
- Month-over-month ESTU membership ∩ size / ESTU size > 95% (i.e., < 5% churn)
- After buffer, the typical ESTU has ~2,500–3,000 members (slightly larger than strict top-3000 because of the hysteresis tail)
- Names that bounce between rank 2,900 and 3,100 should stay in ESTU continuously

## §12. STAGE 7 — Persist deliverable

Write the assembled panel to `data/out/estu_monthly.parquet` with the schema from §4.

Compression: `zstd`. Statistics: `True`.

**Why `data/out/` and not `data/cleaned/`?** This is a derived file built from Sharadar inputs (via the cleaning step). By convention in this repo, anything you derive (as opposed to raw or cleaned vendor data) sits in `data/out/`. The Beta/NLB/NLS guides all read from `data/out/`.

## §13. STAGE 8 — Validation

Run these checks **on the saved deliverable**. ESTU is the upstream input for every USE4 factor; if it's broken, every downstream factor is broken in a correlated way.

### Required checks (all must pass before signing off)

**1. Size of ESTU per date.**
```
Per signal_date t:
    n_estu = count where in_estu == True
Target: 2,300 ≤ n_estu ≤ 3,200 in recent years (post-2010)
Target: 1,500 ≤ n_estu ≤ 2,500 in earlier years (1998–2009)
```

**2. Month-over-month stability.**
```
For consecutive (t, t+1):
    members_t   = set of permatickers in ESTU at t
    members_t1  = set of permatickers in ESTU at t+1
    churn_rate  = |members_t Δ members_t1| / |members_t|
Target: churn_rate < 5% for ≥ 95% of date pairs
Typical: 1–2% with the buffer rule
```

**3. Mega-cap inclusion.**
```
AAPL, MSFT, GOOG, AMZN, NVDA, META, JPM, BRK.B, V, JNJ — all should be in_estu = True
for every date they had a price > 0 in Sharadar.
```

**4. Mcap coverage.**
```
Per signal_date t:
    estu_mcap = sum of mcap for in_estu == True
    total_mcap = sum of mcap for passed_hard_filter == True
    coverage = estu_mcap / total_mcap
Target: coverage ≥ 0.95 (we're aiming for 99% but using total mcap instead of float)
Typical: 0.97–0.99 in recent years
```

**5. Liquidity coverage.**
```
Among in_estu members:
    Median dollar_volume_63d should be > $20M/day in recent years (post-2015)
    10th percentile dollar_volume_63d > $2M/day
Anything below means the liquidity screen is too loose.
```

**6. Size segment proportions.**
```
Per signal_date t, among in_estu members:
    large fraction: ~10-12%   (top ~300 of 3000)
    mid fraction:   ~13-15%   (next ~400 of 3000)
    small fraction: ~73-77%   (rest)
```

**7. No PIT leaks.**
```
For every (permaticker, signal_date t) with in_estu == True:
    daily must have a row for permaticker on date t (or within 5 trading days back)
    daily.filter(date == t).filter(permaticker == p) must have non-null mcap_usd > 0
Spot-check: a stock that delisted on date d should NOT appear in_estu on any signal_date ≥ d+1 month
```

### Optional diagnostic checks

- Time series of `n_estu` — should look smooth, drop slightly in recessions (delisting waves) but not dramatically
- Time series of `large/mid/small` count — should be roughly stable
- Time series of "new entries" per month (`entered_estu = True`) and buffer retentions (`retained_via_buffer = True`) — entries typically 20–50/month with the buffer; would be 100+/month without
- Top 10 by mcap at the latest date — should be the obvious mega-caps (AAPL, MSFT, NVDA, AMZN, GOOG, META, BRK.B, ...)
- Bottom of ESTU at the latest date — should be small-cap stocks with $300M–$1B mcap, not penny stocks
- Industry/sector mix — using GICS sectors (if available), should roughly track S&P 500 sector weights for the large/mid segments

## §14. Master list of ❓ NOT-IN-PDF decisions

ESTU is almost entirely NOT-IN-PDF — USE4 defers to MSCI GIMI for every numeric. The defaults below approximate MSCI IMI principles using Sharadar.

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Float-adjusted vs total mcap | Total mcap (Sharadar lacks float) | Procure free-float data from MSCI or FactSet | If validating against MSCI IMI directly |
| 2 | Allowed exchanges | NYSE, NASDAQ, NYSEMKT | Add BATS, ARCA | If we want broader coverage |
| 3 | Allowed categories | Domestic common stock + REITs | Exclude REITs (separate factor model) | If REIT exposures distort risk model |
| 4 | ADRs included? | No (foreign issuer) | Include with country-flag downstream | If portfolio holds ADRs that need risk forecast |
| 5 | Liquidity metric | ATVR proxy (63d median dollar vol × 252 / mcap): add ≥ 20%, retain ≥ 10% | ATVR (requires float), or longer window, or % days traded | If small-cap segment looks noisy |
| 6 | Liquidity window length | 63 trading days | 21 (1 month), 252 (1 year) | If liquidity oscillates dramatically |
| 7 | Liquidity history requirement | ≥ 42 of 63 days with data | 63 strict, or 21 minimum | Affects how quickly IPOs enter ESTU |
| 8 | Cap target mechanism | Top 3000 by mcap | Cumulative 99% (variable count) | If we want cleaner match to USE4 spec literal text |
| 9 | Buffer add threshold | rank ≤ 3000 | 2500 (tighter), 3500 (looser) | If churn rate is wrong direction |
| 10 | Buffer drop threshold | rank > 3500 | 3000 (no buffer), 4000 (wider buffer) | If too many borderline names flip in/out |
| 11 | Rebalance frequency | Monthly | Quarterly (matches MSCI) | If month-to-month is noisy or expensive |
| 12 | Calendar start | 1998-01-01 | 1995 or 2000 | Earliest date Sharadar has reliable mcap data |
| 13 | Price floor | None (rely on liquidity) | $5 floor (penny-stock filter) | If liquidity screen fails to catch known cases |
| 14 | Sector caps | None | Cap by GICS sector at e.g. 30% | If sector tilts in ESTU distort factor returns |
| 15 | Special handling for new IPOs | 42-day minimum history (implicit) | Explicit wait period of 90 days | If post-IPO names destabilize ESTU |

## §15. Common pitfalls — read this before coding

**Pitfall 1: Survivorship bias.**
- Loading your cleaned ticker table and filtering `isdelisted == False` removes all stocks that delisted *at any point in history*. This is catastrophic survivorship bias.
- Correct approach: a stock is "alive" on date `t` if and only if it has a non-null row in `daily` on date `t`. The `isdelisted` field is the *current* state and useful only as a diagnostic.

**Pitfall 2: Look-ahead in liquidity screen.**
- The 63-day median dollar volume must use only days `≤ t`. Polars' `rolling_median` by default is right-aligned (uses the trailing window), but verify the alignment when chaining with sorts/joins.
- Rule of thumb: at `t`, never reference any row where `daily.date > t`.

**Pitfall 3: Mcap units (Sharadar bug repeat).**
- Sharadar `marketcap` is in **millions** of USD. The mcap threshold logic must use either consistent millions or convert to dollars. Mixing the two gives a factor-of-1000 error in size cutoffs.
- Convert at load time: `mcap_usd = marketcap * 1_000_000`. Then never refer to the raw `marketcap` column again.

**Pitfall 4: Closed-end funds, ETFs, and warrants masquerading as common stock.**
- Sharadar `category` is the authoritative source. Always filter on it.
- A common mistake: filter only by exchange — but NYSE/NASDAQ list plenty of ETFs and CEFs that should NOT be in ESTU.

**Pitfall 5: REIT inclusion is a load-bearing flag.**
- Including REITs adds ~150–200 names to ESTU
- REITs have unusual factor exposures (very high Yield, often high Beta, large Size dispersion)
- USE4 includes REITs; some downstream models exclude them. Be explicit about the choice and document it.

**Pitfall 6: Buffer rule needs warm-start state.**
- On the very first signal_date in the calendar, there's no `prev_estu`. The buffer rule defaults to strict top-N on month 1, then activates on month 2.
- If you restart the pipeline mid-history (e.g., to add new data without rebuilding from 1998), you'll lose the buffer state. The clean fix is to always rebuild from `CALENDAR_START` — it's cheap (~330 dates).

**Pitfall 7: Multiple share classes (GOOG/GOOGL, BRK.A/BRK.B).**
- Sharadar gives each share class its own permaticker. Each permaticker has its own mcap (proportional to that class's share count).
- USE4 / MSCI IMI typically includes both classes if both are independently liquid; they appear as two ESTU members.
- Don't aggregate share classes into one parent issuer for ESTU purposes — keep them separate. The factor model will give them similar Beta/Size scores naturally.

**Pitfall 8: ESTU is the *estimation* universe, not the *holding* universe.**
- This is just the set used for **factor-return estimation** and standardization moments
- The risk model will still produce forecasts for stocks outside ESTU (the *coverage universe*)
- Don't confuse "which stocks the model has a forecast for" with "which stocks are in ESTU"

**Pitfall 9: Liquidity using `closeadj` instead of `closeunadj`.**
- `closeadj` is adjusted for splits and dividends — multiplying it by `volume` (which is unadjusted shares) gives nonsense
- Use `closeunadj * volume` for dollar volume
- This is one of the easiest bugs to introduce and one of the hardest to spot in validation

**Pitfall 10: Don't tune ESTU parameters against factor performance.**
- It's tempting to tweak the buffer thresholds until your favorite factor's t-stat improves
- This is in-sample data mining and produces fragile risk models
- The ESTU spec should be set **once** based on first principles (MSCI IMI), then frozen. Any tuning happens at the descriptor level, not the universe level.

**Pitfall 11: Stale cleaned ticker data.**
- If your cleaned ticker table was built months ago and a recently-IPO'd name isn't in it, that name gets dropped from `tickers` join → null `category` → fails `passed_hard_filter`
- Always check `tickers.shape[0]` vs `daily["permaticker"].n_unique()` — they should be very close
- Rebuild your cleaned ticker table before each ESTU build if Sharadar has been refreshed

## §16. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/estu_monthly.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §4
2. ✅ All 7 required validation checks in §13 pass within tolerance
3. ✅ ESTU has 2,300–3,200 names per date in recent years, stable over time
4. ✅ Month-over-month membership churn < 5% for ≥ 95% of date pairs
5. ✅ Top-10 by mcap on the latest date matches the well-known mega-caps
6. ✅ Mcap coverage (ESTU sum / total sum) ≥ 95% per date
7. ✅ No PIT violations — every in_estu==True row has live price data on signal_date
8. ✅ All ❓ NOT-IN-PDF decisions in §14 are documented in the build script
9. ✅ The file is consumable by `beta/`, `nlb/`, `nls/` guide notebooks without modification

If all 9 are satisfied, ESTU is risk-model-ready and serves as the foundational input for **every** USE4 style factor.

---

### Why this notebook exists

Every USE4 style descriptor reads `estu_monthly.parquet` and uses `in_estu == True` to filter the cross-section that:
1. Provides the cap-weighted-mean and equal-weighted-std for descriptor standardization
2. Provides the constituent set for the cap-weighted market benchmark (Beta input)
3. Provides the sample for sqrt-mcap-weighted orthogonalization regressions (NLB, NLS, RSV)
4. Provides the sample for the cross-sectional WLS regression that backs out factor returns

If ESTU is wrong, all four of those steps are wrong, in a correlated way, for every factor in the model. That's why this spec sits separately and is built first.